In [ ]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder

from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor
)

In [ ]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [ ]:
test_ids = test['Index']

In [ ]:
train.drop('Index', axis=1, inplace=True)
test.drop('Index', axis=1, inplace=True)

In [ ]:
for df in [train, test]:

    df[['hour', 'minute']] = (
        df['timestamp']
        .str.split(':', expand=True)
        .astype(int)
    )

    df['is_peak_hour'] = (
        df['hour'].isin([7,8,9,17,18,19])
    ).astype(int)

    df['is_night'] = (
        ((df['hour'] >= 22) |
         (df['hour'] <= 5))
    ).astype(int)

    df['hour_bin'] = pd.cut(
        df['hour'],
        bins=[0,6,12,18,24],
        labels=False,
        include_lowest=True
    )

In [ ]:
train['RoadType'] = train['RoadType'].fillna(
    train['RoadType'].mode()[0]
)

In [ ]:
train['Weather'] = train['Weather'].fillna(
    train['Weather'].mode()[0]
)

test['Weather'] = test['Weather'].fillna(
    train['Weather'].mode()[0]
)

In [ ]:
train['Temperature'] = train['Temperature'].fillna(
    train['Temperature'].median()
)

test['Temperature'] = test['Temperature'].fillna(
    train['Temperature'].median()
)

In [ ]:
cat_cols = [
    'RoadType',
    'LargeVehicles',
    'Landmarks',
    'Weather'
]

In [ ]:
for col in cat_cols:

    le = LabelEncoder()

    combined = pd.concat([
        train[col],
        test[col]
    ])

    le.fit(combined)

    train[col] = le.transform(train[col])

    test[col] = le.transform(test[col])

In [ ]:
geo_mean = train.groupby(
    'geohash'
)['demand'].mean()

In [ ]:
train['geohash_te'] = (
    train['geohash']
    .map(geo_mean)
)

test['geohash_te'] = (
    test['geohash']
    .map(geo_mean)
)

In [ ]:
global_mean = train['demand'].mean()

test['geohash_te'] = test['geohash_te'].fillna(global_mean)

In [ ]:
for df in [train, test]:

    df['hour_sin'] = np.sin(
        2 * np.pi * df['hour'] / 24
    )

    df['hour_cos'] = np.cos(
        2 * np.pi * df['hour'] / 24
    )

    df['minute_sin'] = np.sin(
        2 * np.pi * df['minute'] / 60
    )

    df['minute_cos'] = np.cos(
        2 * np.pi * df['minute'] / 60
    )

In [ ]:
for df in [train, test]:

    df['lane_pressure'] = (
        df['NumberofLanes']
        /
        (df['RoadType'] + 1)
    )

    df['vehicle_lane_interaction'] = (
        df['LargeVehicles']
        *
        df['NumberofLanes']
    )

    df['temp_hour_interaction'] = (
        df['Temperature']
        *
        df['hour']
    )

    df['day_hour_interaction'] = (
        df['day']
        *
        df['hour']
    )

    df['weather_hour_interaction'] = (
        df['Weather']
        *
        df['hour']
    )

In [ ]:
train.drop(
    ['geohash', 'timestamp'],
    axis=1,
    inplace=True
)

test.drop(
    ['geohash', 'timestamp'],
    axis=1,
    inplace=True
)

In [ ]:
original_train = pd.read_csv('train.csv')
original_test = pd.read_csv('test.csv')

In [ ]:
geo_mean = (
    original_train
    .groupby('geohash')['demand']
    .mean()
)

road_mean = (
    original_train
    .groupby('RoadType')['demand']
    .mean()
)

road_lane_mean = (
    original_train
    .groupby(
        ['RoadType','NumberofLanes']
    )['demand']
    .mean()
)

time_mean = (
    original_train
    .groupby('timestamp')['demand']
    .mean()
)

In [ ]:
original_train['geo_mean'] = (
    original_train['geohash']
    .map(geo_mean)
)

original_test['geo_mean'] = (
    original_test['geohash']
    .map(geo_mean)
)

original_train['road_mean'] = (
    original_train['RoadType']
    .map(road_mean)
)

original_test['road_mean'] = (
    original_test['RoadType']
    .map(road_mean)
)

original_train['road_lane_mean'] = (
    original_train
    .set_index(['RoadType','NumberofLanes'])
    .index
    .map(road_lane_mean)
)

original_test['road_lane_mean'] = (
    original_test
    .set_index(['RoadType','NumberofLanes'])
    .index
    .map(road_lane_mean)
)

original_train['time_mean'] = (
    original_train['timestamp']
    .map(time_mean)
)

original_test['time_mean'] = (
    original_test['timestamp']
    .map(time_mean)
)

In [ ]:
overall_mean = original_train['demand'].mean()

for col in ['geo_mean', 'road_mean', 'road_lane_mean', 'time_mean']:
    original_train[col] = original_train[col].fillna(overall_mean)
    original_test[col] = original_test[col].fillna(overall_mean)

In [ ]:
train['geo_mean'] = original_train['geohash'].map(
    original_train.groupby('geohash')['demand'].mean()
)

test['geo_mean'] = original_test['geohash'].map(
    original_train.groupby('geohash')['demand'].mean()
)

train['road_mean'] = original_train['RoadType'].map(
    original_train.groupby('RoadType')['demand'].mean()
)

test['road_mean'] = original_test['RoadType'].map(
    original_train.groupby('RoadType')['demand'].mean()
)

road_lane_mean = (
    original_train
    .groupby(['RoadType','NumberofLanes'])['demand']
    .mean()
)

train['road_lane_mean'] = list(
    zip(original_train['RoadType'], original_train['NumberofLanes'])
)
train['road_lane_mean'] = train['road_lane_mean'].map(road_lane_mean)

test['road_lane_mean'] = list(
    zip(original_test['RoadType'], original_test['NumberofLanes'])
)
test['road_lane_mean'] = test['road_lane_mean'].map(road_lane_mean)

time_mean = (
    original_train
    .groupby('timestamp')['demand']
    .mean()
)

train['time_mean'] = original_train['timestamp'].map(time_mean)
test['time_mean'] = original_test['timestamp'].map(time_mean)

In [ ]:
overall_mean = original_train['demand'].mean()

for col in ['geo_mean', 'road_mean', 'road_lane_mean', 'time_mean']:
    train[col] = train[col].fillna(overall_mean)
    test[col] = test[col].fillna(overall_mean)

In [ ]:
print(train.columns.tolist())
print(train.shape)
print(test.shape)

In [ ]:
features = [col for col in train.columns if col != 'demand']

X = train[features]
y = train['demand']

X_test = test[features]

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=500,
    max_depth=25,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

rf.fit(X, y)

rf_preds = rf.predict(X_test)

In [ ]:
from sklearn.ensemble import ExtraTreesRegressor

et = ExtraTreesRegressor(
    n_estimators=500,
    max_depth=25,
    random_state=42,
    n_jobs=-1
)

et.fit(X, y)

et_preds = et.predict(X_test)

In [ ]:
final_preds = (
    0.75 * rf_preds +
    0.25 * et_preds
)

In [ ]:
submission = pd.DataFrame({
    'Index': range(len(test)),
    'demand': final_preds
})

submission.to_csv('submission_geo_stats.csv', index=False)

print(submission.head())
print(submission.shape)